<a href="https://colab.research.google.com/github/basit-02/ML-01_Starter/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/basit-02/ML-01_Starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

This queue ranks pages for the Lane 2 editorial review based on directional decision-support scores. The model flags pages where observed engagement metrics and content age suggest decay. The reason codes translate the Logistic Regression coefficients into specific, observed page conditions to guide human editors.

In [1]:
import pandas as pd
import numpy as np

# Assuming test_df, X_test, and the fitted lr_model are loaded from Week 6 outputs
test_df['ml_score'] = lr_model.predict_proba(X_test)[:, 1]

# Sort by the highest probability of decay
action_queue = test_df.sort_values(by='ml_score', ascending=False).copy()

# Generate text-based reason codes based on dominant features
def generate_reason_code(row):
    reasons = []
    if row['days_since_last_update'] > 365:
        reasons.append("Observed stale date (>1 year)")
    if row['adjusted_avg_position'] > 20:
        reasons.append("Low observed SERP position")
    if row['ga4_total_engagement_sec'] < 30:
        reasons.append("Low user engagement measured")

    if not reasons:
        return "Baseline decay threshold met"
    return " compounding with ".join(reasons)

action_queue['reason_code'] = action_queue.apply(generate_reason_code, axis=1)

# Display the top 5 actions for the playbook
display(action_queue[['url', 'ml_score', 'reason_code']].head(5))

NameError: name 'lr_model' is not defined

## 2. Intended use and limits

This model provides directional decision-support for human editors triaging the content refresh queue. It does not predict Google's algorithm. The scores represent patterns observed in the March 2026 dataset.   

Known Limits:
*   Missing avg_position values are aggressively imputed to 100.0, which may artificially inflate the decay risk for unindexed pages.

*   The is_declining_proxy is derived purely from GSC clicks; pages with seasonal traffic drops may be mischaracterized as decaying.

In [ ]:
# Code to validate data slice boundaries to establish limits
date_min = action_queue['date'].min() if 'date' in action_queue.columns else "March 2026"
date_max = action_queue['date'].max() if 'date' in action_queue.columns else "March 2026"
total_pages = len(action_queue)
base_rate = action_queue['is_declining_proxy'].mean()

print(f"Playbook Bounds: Validated on {total_pages} page-days.")
print(f"Timeframe: {date_min} to {date_max}.")
print(f"Observed Base Rate: {base_rate:.4f} (Highly skewed).")

## 3. Human review + the no-go list

Human Review Requirements:
*   Editors must cross-reference the ml_score with the handwritten product flags (health_score, quick_win).
*   Editors must physically review the page formatting to ensure the issue is not a broken UI element rather than stale content.

The No-Go List (Strictly Do Not Automate):
*   Automated deletion of pages based on high ml_score probabilities.
*   Generative AI automated rewrites published without a human editor in the loop.

In [ ]:
# Expose the handwritten product flags alongside the model score for manual review
human_review_cols = ['url', 'ml_score', 'health_score', 'quick_win', 'reason_code']

missing_cols = [col for col in human_review_cols if col not in action_queue.columns]
if not missing_cols:
    review_view = action_queue[human_review_cols].head(10)
    display(review_view)
else:
    print(f"Missing columns for human review view: {missing_cols}")

## 4. Monitoring / retrain triggers

The model must be retrained if the underlying data distribution shifts significantly from the March 2026 training snapshot.

Retrain Triggers:
*   The base rate of is_declining_proxy shifts by more than 5% month-over-month.
*   The median days_since_last_update across the warehouse drops by more than 30%, indicating a massive site-wide refresh that invalidates current coefficients.

In [ ]:
# Record baseline metrics for future drift monitoring
monitoring_baseline = {
    "target_base_rate": float(action_queue['is_declining_proxy'].mean()),
    "median_days_stale": float(action_queue['days_since_last_update'].median()),
    "median_impressions": float(action_queue['gsc_impressions'].median())
}

import json
print("Baseline metrics saved for future monitoring:")
print(json.dumps(monitoring_baseline, indent=2))

## 5. Exports for the paper

The following cell exports the ranked action queue to the outputs directory. This file is explicitly ignored by git via the CI leak-guard to protect raw data. Validation visualizations are saved to the figures directory for inclusion in the Week 8 research paper.

In [ ]:
import os
import matplotlib.pyplot as plt

# Ensure directories exist
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# Export the action queue
action_queue.to_csv('work/outputs/action_queue.csv', index=False)
print("Action queue exported to work/outputs/action_queue.csv")

# Generate and export a generic metric figure (e.g., Score Distribution)
plt.figure(figsize=(8, 4))
plt.hist(action_queue['ml_score'], bins=50, color='blue', alpha=0.7)
plt.title("Distribution of Logistic Regression Scores")
plt.xlabel("Predicted Probability of Decay")
plt.ylabel("Page Count")
plt.tight_layout()

plt.savefig('work/figures/score_distribution.png')
print("Score distribution chart exported to work/figures/score_distribution.png")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.